# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [1]:
%pip install iso3166


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [2]:
# %pip install --upgrade plotly

### Import Statements

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# These might be helpful:
from iso3166 import countries
from datetime import datetime, timedelta

# Added these later
import plotly.graph_objects as go
import calendar

### Notebook Presentation

In [4]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [5]:
df_data = pd.read_csv('mission_launches.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`? 
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [6]:
print(df_data.shape)

(4324, 9)


In [7]:
# how many rows and columns are in the dataset?
# what are the column names and data types?
print(df_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 4324 entries, 0 to 4323
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Unnamed: 0.1    4324 non-null   int64
 1   Unnamed: 0      4324 non-null   int64
 2   Organisation    4324 non-null   str  
 3   Location        4324 non-null   str  
 4   Date            4324 non-null   str  
 5   Detail          4324 non-null   str  
 6   Rocket_Status   4324 non-null   str  
 7   Price           964 non-null    str  
 8   Mission_Status  4324 non-null   str  
dtypes: int64(2), str(7)
memory usage: 304.2 KB
None


In [8]:
# what are the first few rows of the dataset?
print(df_data.head())

   Unnamed: 0.1  Unnamed: 0 Organisation  \
0             0           0       SpaceX   
1             1           1         CASC   
2             2           2       SpaceX   
3             3           3    Roscosmos   
4             4           4          ULA   

                                            Location  \
0         LC-39A, Kennedy Space Center, Florida, USA   
1  Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...   
2                      Pad A, Boca Chica, Texas, USA   
3       Site 200/39, Baikonur Cosmodrome, Kazakhstan   
4           SLC-41, Cape Canaveral AFS, Florida, USA   

                         Date                                        Detail  \
0  Fri Aug 07, 2020 05:12 UTC  Falcon 9 Block 5 | Starlink V1 L9 & BlackSky   
1  Thu Aug 06, 2020 04:01 UTC           Long March 2D | Gaofen-9 04 & Q-SAT   
2  Tue Aug 04, 2020 23:57 UTC            Starship Prototype | 150 Meter Hop   
3  Thu Jul 30, 2020 21:25 UTC  Proton-M/Briz-M | Ekspress-80 & Ekspress-103   
4  

In [9]:



# are there any NaN values in the dataset? If so, how many are there in each column?
print(df_data.isna().sum())


Unnamed: 0.1         0
Unnamed: 0           0
Organisation         0
Location             0
Date                 0
Detail               0
Rocket_Status        0
Price             3360
Mission_Status       0
dtype: int64


In [10]:
# are there any duplicate rows in the dataset?
print(df_data.duplicated().sum())

0


## Data Cleaning - Check for Missing Values and Duplicates

Consider removing columns containing junk data. 

In [11]:
# Remove unnecessary index columns and columns with mostly NaN values
df_data = df_data.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])

# Display the cleaned dataframe
print(df_data.head())
print(f"\nDataframe shape after cleaning: {df_data.shape}")
print(f"\nRemaining columns: {df_data.columns.tolist()}")

  Organisation                                           Location  \
0       SpaceX         LC-39A, Kennedy Space Center, Florida, USA   
1         CASC  Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...   
2       SpaceX                      Pad A, Boca Chica, Texas, USA   
3    Roscosmos       Site 200/39, Baikonur Cosmodrome, Kazakhstan   
4          ULA           SLC-41, Cape Canaveral AFS, Florida, USA   

                         Date                                        Detail  \
0  Fri Aug 07, 2020 05:12 UTC  Falcon 9 Block 5 | Starlink V1 L9 & BlackSky   
1  Thu Aug 06, 2020 04:01 UTC           Long March 2D | Gaofen-9 04 & Q-SAT   
2  Tue Aug 04, 2020 23:57 UTC            Starship Prototype | 150 Meter Hop   
3  Thu Jul 30, 2020 21:25 UTC  Proton-M/Briz-M | Ekspress-80 & Ekspress-103   
4  Thu Jul 30, 2020 11:50 UTC                    Atlas V 541 | Perseverance   

  Rocket_Status  Price Mission_Status  
0  StatusActive   50.0        Success  
1  StatusActive  29.75        

## Descriptive Statistics

# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [12]:
# Count launches by organisation
launches_by_org = df_data['Organisation'].value_counts().head(10)

# Create a bar chart
fig = px.bar(
	x=launches_by_org.index,
	y=launches_by_org.values,
	labels={'x': 'Organisation', 'y': 'Number of Launches'},
	title='Top 10 Organisations by Number of Space Mission Launches',
	color=launches_by_org.values,
	color_continuous_scale='Viridis'
)

fig.update_layout(xaxis_tickangle=-45, height=500, showlegend=False)
fig.show()

# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned? 

In [13]:
rocket_status_counts = df_data['Rocket_Status'].value_counts()

fig = px.pie(
    values=rocket_status_counts.values,
    names=rocket_status_counts.index,
    title='Active vs Retired Rockets',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textinfo='percent+label')
fig.show()

# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [14]:
mission_status_counts = df_data['Mission_Status'].value_counts()

fig = px.pie(
    values=mission_status_counts.values,
    names=mission_status_counts.index,
    title='Distribution of Mission Status',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textinfo='percent+label')
fig.show()

# How Expensive are the Launches? 

Create a histogram and visualise the distribution. The price column is given in USD millions (careful of missing values). 

In [15]:
fig = px.histogram(
    df_data.dropna(subset=['Price']),
    x='Price',
    nbins=50,
    title='Distribution of Launch Prices (USD Millions)',
    labels={'Price': 'Price (USD Millions)', 'count': 'Number of Launches'},
    color_discrete_sequence=['steelblue']
)

fig.update_layout(bargap=0.1)
fig.show()

# Use a Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Experiment with [plotly's available colours](https://plotly.com/python/builtin-colorscales/). I quite like the sequential colour `matter` on this map. 
* You'll need to extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

You'll need to use a 3 letter country code for each country. You might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


You can use the iso3166 package to convert the country names to Alpha3 format.

In [16]:
# Step 1: Extract country from the last segment of the Location string
df_data['country'] = df_data['Location'].apply(lambda x: x.split(', ')[-1].strip())

# Step 2: Remap country names that no longer exist, are geographical references,
# or use common names instead of official UN names (required by iso3166).
# Note: remaps are applied in one pass, so chained remaps won't work —
# each source key must map directly to its final UN name.
country_remaps = {
    # Historical / no longer existing
    'Russia': 'Russian Federation',
    'Barents Sea': 'Russian Federation',
    # Geographic references → owning country
    'New Mexico': 'USA',
    'Pacific Missile Range Facility': 'USA',
    'Gran Canaria': 'USA',
    'Pacific Ocean': 'USA',
    'Yellow Sea': 'China',
    # Must map directly to official UN name (not via 'Iran' — replace() is single-pass)
    'Shahrud Missile Test Site': 'Iran, Islamic Republic of',
    'Iran': 'Iran, Islamic Republic of',
    'North Korea': "Korea, Democratic People's Republic of",
    'South Korea': 'Korea, Republic of',
}
df_data['country'] = df_data['country'].replace(country_remaps)

# Step 3: Convert country names to ISO 3166 Alpha-3 codes
def to_alpha3(country_name):
    try:
        return countries.get(country_name).alpha3
    except KeyError:
        return None

df_data['iso_alpha3'] = df_data['country'].apply(to_alpha3)

unresolved = df_data[df_data['iso_alpha3'].isna()]['country'].unique()
print(f"Unresolved countries: {unresolved}")

Unresolved countries: <StringArray>
[]
Length: 0, dtype: str


In [17]:
# Step 4: Count launches per country (drop any unresolved rows)
launches_by_country = (
    df_data
    .dropna(subset=['iso_alpha3'])
    .groupby(['country', 'iso_alpha3'])
    .size()
    .reset_index(name='launches')
)

# Step 5: Plot choropleth map
fig = px.choropleth(
    launches_by_country,
    locations='iso_alpha3',
    color='launches',
    hover_name='country',
    color_continuous_scale='matter',
    title='Number of Space Mission Launches by Country',
    labels={'launches': 'Number of Launches'}
)

fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

# Use a Choropleth Map to Show the Number of Failures by Country


In [18]:
# Build a summary table: total missions, failures, and failure rate per country
total_by_country = (
    df_data.dropna(subset=['iso_alpha3'])
    .groupby(['country', 'iso_alpha3'])
    .size()
    .reset_index(name='missions')
)

failures_by_country = (
    df_data[df_data['Mission_Status'] != 'Success']
    .dropna(subset=['iso_alpha3'])
    .groupby(['country', 'iso_alpha3'])
    .size()
    .reset_index(name='failures')
)

# Merge and calculate failure rate
country_stats = total_by_country.merge(failures_by_country, on=['country', 'iso_alpha3'], how='left')
country_stats['failures'] = country_stats['failures'].fillna(0).astype(int)
country_stats['failure_rate'] = (country_stats['failures'] / country_stats['missions']).round(3)

country_stats.sort_values('missions', ascending=False)

,country,iso_alpha3,missions,failures,failure_rate
13,Russian Federation,RUS,1398,93,0.07
14,USA,USA,1387,166,0.12
8,Kazakhstan,KAZ,701,93,0.13
3,France,FRA,303,18,0.06
2,China,CHN,269,25,0.09
7,Japan,JPN,126,13,0.10
4,India,IND,76,13,0.17
5,"Iran, Islamic Republic of",IRN,14,9,0.64
12,New Zealand,NZL,13,2,0.15
6,Israel,ISR,11,2,0.18


In [19]:
# Choropleth of failure RATE (failures per mission) — avoids raw count bias
# e.g. USSR had many failures but also many missions; rate tells the real story
fig = px.choropleth(
    country_stats,
    locations='iso_alpha3',
    color='failure_rate',
    hover_name='country',
    hover_data={'missions': True, 'failures': True, 'failure_rate': ':.1%'},
    color_continuous_scale='RdYlGn_r',
    title='Space Mission Failure Rate by Country (Failures per Mission)',
    labels={'failure_rate': 'Failure Rate'}
)

fig.update_coloraxes(colorbar_tickformat='.0%')
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status. 

In [20]:
# Sunburst hierarchy: Country → Organisation → Mission Status
# Each slice size = number of missions at that level

fig = px.sunburst(
    df_data,
    path=['country', 'Organisation', 'Mission_Status'],  # outer → inner drill-down
    title='Space Missions by Country, Organisation, and Mission Status',
    color='Mission_Status',
    color_discrete_map={
        'Success': '#2ecc71',
        'Failure': '#e74c3c',
        'Partial Failure': '#e67e22',
        'Prelaunch Failure': '#c0392b',
    },
    height=700,
)

fig.update_traces(textinfo='label+percent entry')
fig.show()

# Analyse the Total Amount of Money Spent by Organisation on Space Missions

In [21]:
# Price column has ~3360 NaNs — we only analyse orgs that have at least some price data.
# Convert Price to numeric (it's stored as str) then sum per organisation.
df_data['Price'] = pd.to_numeric(df_data['Price'], errors='coerce')

total_spend = (
    df_data.dropna(subset=['Price'])
    .groupby('Organisation')['Price']
    .sum()
    .reset_index(name='total_spend_usd_m')
    .sort_values('total_spend_usd_m', ascending=False)
)

# Preview the table
total_spend.head(10)

,Organisation,total_spend_usd_m
14,NASA,"61,200.00"
0,Arianespace,"16,345.00"
20,ULA,"14,798.00"
2,CASC,"6,340.26"
19,SpaceX,"5,444.00"
15,Northrop,"3,930.00"
12,MHI,"3,532.50"
8,ISRO,"2,177.00"
21,US Air Force,"1,550.92"
22,VKS RF,"1,548.90"


In [22]:
# Bar chart — top organisations by total spend (USD millions)
# Only orgs with price data are shown; NaN rows are excluded from the sum
fig = px.bar(
    total_spend.head(15),
    x='Organisation',
    y='total_spend_usd_m',
    title='Total Amount Spent on Space Missions by Organisation (USD Millions)',
    labels={'total_spend_usd_m': 'Total Spend (USD Millions)', 'Organisation': 'Organisation'},
    color='total_spend_usd_m',
    color_continuous_scale='Viridis',
)

fig.update_layout(xaxis_tickangle=-45, height=500, showlegend=False)
fig.show()

# Analyse the Amount of Money Spent by Organisation per Launch

In [23]:
# Average cost per launch = total spend / number of priced launches
# We use only rows with a Price value — launches without a price are excluded
# (using all launches in the denominator would artificially deflate the average)

spend_per_launch = (
    df_data.dropna(subset=['Price'])
    .groupby('Organisation')
    .agg(
        total_spend=('Price', 'sum'),
        priced_launches=('Price', 'count'),
    )
    .reset_index()
)

spend_per_launch['avg_cost_per_launch'] = (
    spend_per_launch['total_spend'] / spend_per_launch['priced_launches']
).round(2)

spend_per_launch = spend_per_launch.sort_values('avg_cost_per_launch', ascending=False)

# Preview table
spend_per_launch.head(10)

,Organisation,total_spend,priced_launches,avg_cost_per_launch
14,NASA,"61,200.00",136,450.00
1,Boeing,"1,241.00",7,177.29
0,Arianespace,"16,345.00",96,170.26
20,ULA,"14,798.00",98,151.00
7,ILS,"1,320.00",13,101.54
12,MHI,"3,532.50",37,95.47
13,Martin Marietta,721.40,9,80.16
21,US Air Force,"1,550.92",26,59.65
9,JAXA,168.00,3,56.00
19,SpaceX,"5,444.00",99,54.99


In [24]:
# Bar chart — average cost per launch by organisation (top 15)
# Orgs with only 1-2 priced launches can skew this heavily, so hover shows launch count for context
fig = px.bar(
    spend_per_launch.head(15),
    x='Organisation',
    y='avg_cost_per_launch',
    hover_data={'priced_launches': True, 'total_spend': True},
    title='Average Cost per Launch by Organisation (USD Millions)',
    labels={
        'avg_cost_per_launch': 'Avg Cost per Launch (USD Millions)',
        'priced_launches': 'Priced Launches',
        'total_spend': 'Total Spend (USD Millions)',
    },
    color='avg_cost_per_launch',
    color_continuous_scale='Viridis',
)

fig.update_layout(xaxis_tickangle=-45, height=500, showlegend=False)
fig.show()

# Chart the Number of Launches per Year

In [25]:
# Date is stored as a string e.g. "Fri Aug 07, 2020 05:12 UTC"
# Use format='mixed' to handle any minor inconsistencies across rows
df_data['Date'] = pd.to_datetime(df_data['Date'], format='mixed', utc=True)
df_data['year'] = df_data['Date'].dt.year

# Count launches per year
launches_per_year = df_data.groupby('year').size().reset_index(name='launches')

fig = px.bar(
    launches_per_year,
    x='year',
    y='launches',
    title='Number of Space Mission Launches per Year',
    labels={'year': 'Year', 'launches': 'Number of Launches'},
    color='launches',
    color_continuous_scale='Viridis',
)

fig.update_layout(xaxis=dict(dtick=5), height=500, showlegend=False)
fig.show()

# Chart the Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart. 

In [26]:
# import plotly.graph_objects as go
# imported at the top of the notebook, for clean import block

# Build a year-month period index and count launches per month
# Date was already parsed to datetime in the launches-per-year cell
df_data['year_month'] = df_data['Date'].dt.to_period('M')

launches_mom = (
    df_data.groupby('year_month')
    .size()
    .reset_index(name='launches')
)

# Convert Period to timestamp for Plotly (Periods aren't directly plottable)
launches_mom['year_month_ts'] = launches_mom['year_month'].dt.to_timestamp()

# 6-month rolling average — smooths short-term noise to reveal long-run trend
launches_mom['rolling_avg'] = launches_mom['launches'].rolling(window=6, center=True).mean()

# Identify the single month with the all-time highest launch count
peak = launches_mom.loc[launches_mom['launches'].idxmax()]
print(f"Highest ever: {peak['launches']} launches in {peak['year_month']}")

# Build chart: bar for raw monthly counts + line for rolling average
fig = go.Figure()

fig.add_trace(go.Bar(
    x=launches_mom['year_month_ts'],
    y=launches_mom['launches'],
    name='Monthly Launches',
    marker_color='steelblue',
    opacity=0.6,
))

fig.add_trace(go.Scatter(
    x=launches_mom['year_month_ts'],
    y=launches_mom['rolling_avg'],
    name='6-Month Rolling Avg',
    line=dict(color='crimson', width=2),
))

fig.update_layout(
    title='Space Mission Launches Month-on-Month with 6-Month Rolling Average',
    xaxis_title='Date',
    yaxis_title='Number of Launches',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500,
)

fig.show()

Highest ever: 18 launches in 1971-12


/var/folders/tc/pjb8rnpn2lg96cjyfl1k9zt00000gn/T/ipykernel_91413/2376061069.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_data['year_month'] = df_data['Date'].dt.to_period('M')


# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [27]:
# import calendar 
# imported at the top of the notebook, for clean import block
# we need calendar.month_abbr to convert month numbers to names for better readability in the next steps

# Extract month number and name — aggregate across ALL years to find seasonal patterns
df_data['month'] = df_data['Date'].dt.month
df_data['month_name'] = df_data['Date'].dt.month.apply(lambda m: calendar.month_abbr[m])

launches_by_month = (
    df_data.groupby(['month', 'month_name'])
    .size()
    .reset_index(name='launches')
    .sort_values('month')  # keep Jan→Dec order
)

# Identify best and worst months
best  = launches_by_month.loc[launches_by_month['launches'].idxmax()]
worst = launches_by_month.loc[launches_by_month['launches'].idxmin()]
print(f"Most launches:   {best['month_name']}  ({best['launches']})")
print(f"Fewest launches: {worst['month_name']} ({worst['launches']})")

fig = px.bar(
    launches_by_month,
    x='month_name',
    y='launches',
    title='Total Space Mission Launches by Month (All Years)',
    labels={'month_name': 'Month', 'launches': 'Total Launches'},
    color='launches',
    color_continuous_scale='Blues',
    category_orders={'month_name': list(calendar.month_abbr[1:])},  # enforce Jan→Dec, otherwise it will alpha
)

fig.update_layout(height=450, showlegend=False)
fig.show()

Most launches:   Dec  (450)
Fewest launches: Jan (268)


# How has the Launch Price varied Over Time? 

Create a line chart that shows the average price of rocket launches over time. 

In [28]:
# Average price per year — only rows with a Price value contribute
# (Price was converted to numeric in the total-spend cell)
avg_price_per_year = (
    df_data.dropna(subset=['Price'])
    .groupby('year')['Price']
    .mean()
    .reset_index(name='avg_price')
)

fig = px.line(
    avg_price_per_year,
    x='year',
    y='avg_price',
    title='Average Launch Price Over Time (USD Millions)',
    labels={'year': 'Year', 'avg_price': 'Avg Price (USD Millions)'},
    markers=True,
)

fig.update_layout(height=450)
fig.show()

******

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991. 

## Create a Plotly Pie Chart comparing the total number of launches of the USSR and the USA

Hint: Remember to include former Soviet Republics like Kazakhstan when analysing the total number of launches. 

In [29]:
# The USSR dissolved in 1991, so we split into two eras for a fair comparison:
#   Cold War:  start of dataset → 1991 (USA vs USSR)
#   Post-Cold War: 1992 → present (USA vs Russia, which inherited the Soviet programme)
#
# USSR/Soviet launch countries: Russia (Russian Federation) + Kazakhstan (Baikonur)

ussr_countries = {'Russian Federation', 'Kazakhstan'}

def superpower(country):
    if country == 'USA':
        return 'USA'
    elif country in ussr_countries:
        return 'USSR / Russia'
    return 'Other'

df_data['superpower'] = df_data['country'].apply(superpower)

# Split by era — year column was created in the launches-per-year cell
cold_war      = df_data[(df_data['year'] <= 1991) & (df_data['superpower'] != 'Other')]
post_cold_war = df_data[(df_data['year'] >= 1992) & (df_data['superpower'] != 'Other')]

color_map = {'USA': '#1f77b4', 'USSR / Russia': '#d62728'}

# --- Chart 1: Cold War ---
cw_counts = cold_war.groupby('superpower').size().reset_index(name='launches')
fig1 = px.pie(
    cw_counts,
    values='launches',
    names='superpower',
    title='Cold War Launches (1957–1991): USA vs USSR',
    color='superpower',
    color_discrete_map=color_map,
)
fig1.update_traces(textinfo='percent+label+value')
fig1.show()

# --- Chart 2: Post Cold War ---
pcw_counts = post_cold_war.groupby('superpower').size().reset_index(name='launches')
fig2 = px.pie(
    pcw_counts,
    values='launches',
    names='superpower',
    title='Post-Cold War Launches (1992–Present): USA vs Russia',
    color='superpower',
    color_discrete_map=color_map,
)
fig2.update_traces(textinfo='percent+label+value')
fig2.show()


In [30]:
# All-time total launches for USA vs USSR/Russia (no era filter)
all_time = df_data[df_data['superpower'] != 'Other'].groupby('superpower').size().reset_index(name='launches')

fig = px.bar(
    all_time,
    x='superpower',
    y='launches',
    title='All-Time Total Launches: USA vs USSR / Russia',
    labels={'superpower': '', 'launches': 'Total Launches'},
    color='superpower',
    color_discrete_map={'USA': '#1f77b4', 'USSR / Russia': '#d62728'},
    text='launches',
)

fig.update_traces(textposition='outside')
fig.update_layout(height=450, showlegend=False)
fig.show()


## Create a Chart that Shows the Total Number of Launches Year-On-Year by the Two Superpowers

In [31]:
# Count launches per year per superpower — includes ALL years in the dataset
# superpower column was created in the USSR/USA pie chart cell
yoy = (
    df_data[df_data['superpower'] != 'Other']
    .groupby(['year', 'superpower'])
    .size()
    .reset_index(name='launches')
)

fig = px.line(
    yoy,
    x='year',
    y='launches',
    color='superpower',
    title='Year-on-Year Space Mission Launches: USA vs USSR / Russia',
    labels={'year': 'Year', 'launches': 'Number of Launches', 'superpower': ''},
    color_discrete_map={'USA': '#1f77b4', 'USSR / Russia': '#d62728'},
    markers=True,
)

# Mark the end of the Cold War / USSR dissolution
fig.add_vline(
    x=1991,
    line_dash='dash',
    line_color='grey',
    annotation_text='USSR dissolved (1991)',
    annotation_position='top right',
)

fig.update_layout(height=500, legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()


## Chart the Total Number of Mission Failures Year on Year.

In [32]:
# Failure % per year per superpower = failures / total launches * 100
# Counts all non-Success statuses (Failure, Partial Failure, Prelaunch Failure)

superpower_df = df_data[df_data['superpower'] != 'Other'].copy()

total_yoy = (
    superpower_df
    .groupby(['year', 'superpower'])
    .size()
    .reset_index(name='total')
)

failures_yoy = (
    superpower_df[superpower_df['Mission_Status'] != 'Success']
    .groupby(['year', 'superpower'])
    .size()
    .reset_index(name='failures')
)

failure_rate_yoy = total_yoy.merge(failures_yoy, on=['year', 'superpower'], how='left')
failure_rate_yoy['failures'] = failure_rate_yoy['failures'].fillna(0)
failure_rate_yoy['failure_pct'] = (failure_rate_yoy['failures'] / failure_rate_yoy['total'] * 100).round(1)

fig = px.line(
    failure_rate_yoy,
    x='year',
    y='failure_pct',
    color='superpower',
    title='Year-on-Year Mission Failure Rate (%): USA vs USSR / Russia',
    labels={'year': 'Year', 'failure_pct': 'Failure Rate (%)', 'superpower': ''},
    color_discrete_map={'USA': '#1f77b4', 'USSR / Russia': '#d62728'},
    markers=True,
)

fig.add_vline(
    x=1991,
    line_dash='dash',
    line_color='grey',
    annotation_text='USSR dissolved (1991)',
    annotation_position='top right',
)

fig.update_layout(height=500, legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()


## Chart the Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time? 

In [33]:
# Failure % across ALL organisations globally, per year
# Shows whether the industry as a whole got safer over time

total_per_year = df_data.groupby('year').size().reset_index(name='total')

failures_per_year = (
    df_data[df_data['Mission_Status'] != 'Success']
    .groupby('year')
    .size()
    .reset_index(name='failures')
)

global_failure_rate = total_per_year.merge(failures_per_year, on='year', how='left')
global_failure_rate['failures'] = global_failure_rate['failures'].fillna(0)
global_failure_rate['failure_pct'] = (
    global_failure_rate['failures'] / global_failure_rate['total'] * 100
).round(1)

# 5-year rolling average to smooth year-to-year noise
global_failure_rate['rolling_avg'] = (
    global_failure_rate['failure_pct'].rolling(window=5, center=True).mean()
)

import plotly.graph_objects as go

fig = go.Figure()

# Raw annual failure rate as bars
fig.add_trace(go.Bar(
    x=global_failure_rate['year'],
    y=global_failure_rate['failure_pct'],
    name='Annual Failure %',
    marker_color='#e74c3c',
    opacity=0.5,
))

# 5-year rolling average as a trend line
fig.add_trace(go.Scatter(
    x=global_failure_rate['year'],
    y=global_failure_rate['rolling_avg'],
    name='5-Year Rolling Avg',
    line=dict(color='#2c3e50', width=2),
))

fig.update_layout(
    title='Global Space Mission Failure Rate Over Time (%)',
    xaxis_title='Year',
    yaxis_title='Failure Rate (%)',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)

fig.show()


# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020)

Do the results change if we only look at the number of successful launches? 

In [34]:
# For each year, find which country had the most launches (all missions, up to 2020)
# We use the country column derived from Location in the choropleth cells

df_2020 = df_data[df_data['year'] <= 2020]

# Count launches per year per country, then take the country with the max for each year
all_launches_yoy = (
    df_2020.groupby(['year', 'country'])
    .size()
    .reset_index(name='launches')
)

leading_country = (
    all_launches_yoy
    .loc[all_launches_yoy.groupby('year')['launches'].idxmax()]
    .reset_index(drop=True)
)

fig = px.bar(
    leading_country,
    x='year',
    y='launches',
    color='country',
    title='Leading Country in Space Launches Per Year (All Missions, up to 2020)',
    labels={'year': 'Year', 'launches': 'Launches', 'country': 'Country'},
)

fig.update_layout(height=500)
fig.show()


In [35]:
# Same chart but restricted to SUCCESSFUL launches only
# Does the leading country change when we filter out failures?

successful = df_data[(df_data['year'] <= 2020) & (df_data['Mission_Status'] == 'Success')]

success_yoy = (
    successful.groupby(['year', 'country'])
    .size()
    .reset_index(name='launches')
)

leading_country_success = (
    success_yoy
    .loc[success_yoy.groupby('year')['launches'].idxmax()]
    .reset_index(drop=True)
)

fig = px.bar(
    leading_country_success,
    x='year',
    y='launches',
    color='country',
    title='Leading Country in Space Launches Per Year (Successful Missions Only, up to 2020)',
    labels={'year': 'Year', 'launches': 'Successful Launches', 'country': 'Country'},
)

fig.update_layout(height=500)
fig.show()

# Quick comparison — do the leaders differ between all vs successful?
merged = leading_country[['year','country']].merge(
    leading_country_success[['year','country']],
    on='year', suffixes=('_all', '_success')
)
differs = merged[merged['country_all'] != merged['country_success']]
if differs.empty:
    print("No difference — the leading country is the same every year whether or not we filter to successful launches.")
else:
    print("Years where the leader changes when filtering to successful launches only:")
    print(differs.to_string(index=False))


Years where the leader changes when filtering to successful launches only:
 year country_all    country_success
 1967  Kazakhstan Russian Federation
 1968  Kazakhstan Russian Federation
 1993         USA Russian Federation
 2020       China                USA


# Create a Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

Which organisation was dominant in the 1970s and 1980s? Which organisation was dominant in 2018, 2019 and 2020? 

In [36]:
# For each year, find the organisation with the most launches
dominant_org = (
    df_data.groupby(['year', 'Organisation'])
    .size()
    .reset_index(name='launches')
)

dominant_org = (
    dominant_org
    .loc[dominant_org.groupby('year')['launches'].idxmax()]
    .reset_index(drop=True)
)

fig = px.bar(
    dominant_org,
    x='year',
    y='launches',
    color='Organisation',
    title='Most Active Organisation per Year (by Number of Launches)',
    labels={'year': 'Year', 'launches': 'Launches', 'Organisation': 'Organisation'},
)

fig.update_layout(height=550)
fig.show()

# Directly answer the questions
print("Dominant organisations in the 1970s and 1980s:")
print(dominant_org[dominant_org['year'].between(1970, 1989)][['year','Organisation','launches']].to_string(index=False))

print("\nDominant organisations in 2018, 2019, 2020:")
print(dominant_org[dominant_org['year'].isin([2018, 2019, 2020])][['year','Organisation','launches']].to_string(index=False))


Dominant organisations in the 1970s and 1980s:
 year Organisation  launches
 1970    RVSN USSR        87
 1971    RVSN USSR        93
 1972    RVSN USSR        79
 1973    RVSN USSR        84
 1974    RVSN USSR        83
 1975    RVSN USSR        89
 1976    RVSN USSR        95
 1977    RVSN USSR        97
 1978    RVSN USSR        71
 1979    RVSN USSR        36
 1980    RVSN USSR        40
 1981    RVSN USSR        51
 1982    RVSN USSR        53
 1983    RVSN USSR        46
 1984    RVSN USSR        42
 1985    RVSN USSR        51
 1986    RVSN USSR        49
 1987    RVSN USSR        42
 1988    RVSN USSR        36
 1989    RVSN USSR        26

Dominant organisations in 2018, 2019, 2020:
 year Organisation  launches
 2018         CASC        37
 2019         CASC        27
 2020         CASC        19


Completed by *Claude Code* and *rudil24@gmail.com*
***